# NeuroObfuscator v7.1 — Inference & Evaluation (GGUF)

**Модель**: `gguf_v7/*.gguf` (Qwen2.5-Coder-7B, q4_k_m, LoRA вшиты при экспорте) через **llama.cpp**.
Оцениваем ровно тот артефакт, который будет задеплоен.

**Датасет v7.1 (conditional)**: промпт содержит `Target intensity: X`; интенсивность определяет форму плана:
light = rename+dead_code only, medium = без opaque, heavy = с opaque.

Метрики гейтов:
- JSON parse / schema (>= 95% / 90%)
- **intensity obedience** (поле) и **shape obedience** (контент: light→minimal, medium→no-opaque, heavy→opaque)
- **light purity** (>= 95%)
- diversity: unique orders (info), top non-light order share (<= 45%; данные v7.1 дают 28%)
- semantic pass rate через Node-движок (`engine_bundle_v6.zip` — движок не менялся)

Этапы: probe → quick eval (10) → full eval (750) → semantic → отчёт → demo (3 intensity).

Первая ячейка: llama-cpp-python собирается из исходников (~10-15 мин один раз, wheel кэшируется на Drive по GPU-архитектуре); повторные сессии — секунды.


In [ ]:
# llama-cpp-python: под Colab py3.12/3.13 нет готовых CUDA-wheel'ов -> сборка из исходников.
# Без оптимизаций это 20-30 мин (nvcc собирает ядра для всех GPU-архитектур).
# Оптимизации: (1) CMAKE_CUDA_ARCHITECTURES только под текущий GPU (A100=80, L4=89),
# (2) собранный wheel кэшируется на Drive -> повторные сессии ставятся за секунды.
import os, glob, subprocess, sys, torch

from google.colab import drive
drive.mount('/content/drive')

cap = torch.cuda.get_device_capability(0)
arch = ''.join(map(str, cap))   # A100 -> '80', L4 -> '89'
WHEEL_DIR = f'/content/drive/MyDrive/neuroobfuscator/wheels/{arch}'
os.makedirs(WHEEL_DIR, exist_ok=True)
os.environ['CMAKE_ARGS'] = f'-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES={arch}'
print('GPU:', torch.cuda.get_device_name(0), '| CUDA arch:', arch)

wheels = sorted(glob.glob(f'{WHEEL_DIR}/llama_cpp_python-*.whl'))
if wheels:
    print('ставим из кэша Drive:', wheels[-1])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', wheels[-1]], check=True)
else:
    print('первая сборка из исходников (~10-15 мин с одним arch), wheel сохранится на Drive...')
    subprocess.run([sys.executable, '-m', 'pip', 'wheel', 'llama-cpp-python', '-w', WHEEL_DIR, '-q'], check=True)
    built = sorted(glob.glob(f'{WHEEL_DIR}/llama_cpp_python-*.whl'))
    assert built, 'wheel не собрался'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', built[-1]], check=True)

import llama_cpp
print('llama-cpp-python ready:', llama_cpp.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
ROOT = '/content/drive/MyDrive/neuroobfuscator'
DATA_DIR   = f'{ROOT}/final_v7'
ENGINE_ZIP = f'{ROOT}/engine_bundle_v6.zip'   # движок не менялся с v6.1
OUT_DIR    = f'{ROOT}/eval_results_v7'
os.makedirs(OUT_DIR, exist_ok=True)

# Только gguf_v7: никакой рекурсивной подстановки — иначе можно молча
# подхватить старый gguf_v6 и оценить не ту модель.
ggufs = glob.glob(f'{ROOT}/gguf_v7/*.gguf')
assert ggufs, f'GGUF не найден в {ROOT}/gguf_v7 — залить экспорт из train-ноутбука (ячейка GGUF)'
GGUF_PATH = max(ggufs, key=os.path.getsize)
print('gguf:', GGUF_PATH, '|', round(os.path.getsize(GGUF_PATH)/1e9, 2), 'GB')
print('data:', sorted(os.listdir(DATA_DIR)))


In [ ]:
import time
from llama_cpp import Llama

t0 = time.time()
llm = Llama(
    model_path=GGUF_PATH,
    n_gpu_layers=-1,
    n_ctx=4096,
    verbose=False,
)
print(f'model loaded in {time.time()-t0:.0f}s')


In [ ]:
import json, re, random, time

SYSTEM_PROMPT = 'You are NeuroObfuscator. Given JavaScript code and its AST features, generate an optimal obfuscation plan as a JSON object.\n\nAvailable transformations (apply in this order when enabled):\n1. rename         - Rename local identifiers to hex-like names. Almost always recommended.\n2. string_encode  - Encode string literals. Methods: charcode_array, charcode_concat, hex_escape, unicode_escape. Only enable if string_count > 0.\n3. operator_sub   - Substitute arithmetic/comparison operators (a+b -> a-(-b), a===b -> !(a!==b)). Use when operator_count > 2.\n4. dead_code      - Insert unreachable code blocks. count: 1-5. More complex code tolerates more.\n5. opaque_predicates - Insert always-true/always-false conditions. count: 1-3. Primarily for medium/heavy intensity; may also be used sparingly on light functions when extra diversity is needed.\n\nIntensity guide:\n- light:  cyclomatic_complexity <= 2. Prefer rename + dead_code only.\n- medium: complexity 3-5. Add string_encode and operator_sub if applicable.\n- heavy:  complexity > 5. Use all relevant transforms aggressively.\n\nRules:\n- You MUST honor the requested "Target intensity" when it is provided, even if\n  it differs from what the complexity alone would suggest. Intensity determines\n  the plan shape:\n  light  -> minimal plan: rename + dead_code ONLY (no string_encode,\n            no operator_sub, no opaque_predicates),\n  medium -> moderate plan: rename + dead_code + string_encode/operator_sub\n            when applicable, NO opaque_predicates,\n  heavy  -> aggressive plan: all relevant transforms INCLUDING opaque_predicates.\n- Only include enabled transforms in "order" array.\n- Order MUST follow: rename, string_encode, operator_sub, dead_code, opaque_predicates.\n- Do NOT include a "seed" field in your JSON; the runtime injects the provided seed automatically.\n- Avoid over-bloating small functions.\n\nOutput ONLY valid JSON. No explanations, no markdown.'

ORDER = ['rename', 'string_encode', 'operator_sub', 'dead_code', 'opaque_predicates']

def _complexity_class(cc):
    if cc <= 2: return 'light'
    elif cc <= 5: return 'medium'
    return 'heavy'

def _target_of(instruction):
    m = re.search(r'Target intensity: (light|medium|heavy)', instruction)
    return m.group(1) if m else None

def format_prompt_inst(code, features, seed, target_intensity=None):
    features_json = json.dumps(features, separators=(',', ':'))
    intensity_line = f'Target intensity: {target_intensity}\n' if target_intensity else ''
    return (
        f"[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"=== CODE ===\n{code}\n=== END CODE ===\n\n"
        f"=== AST FEATURES ===\n{features_json}\n=== END AST FEATURES ===\n\n"
        f"complexity_class={_complexity_class(features.get('cyclomatic_complexity', 1))} "
        f"(cyclomatic_complexity={features.get('cyclomatic_complexity', 1)})\n"
        f"{intensity_line}"
        f"seed={seed}\n\n"
        f"Generate the obfuscation plan JSON: [/INST]"
    )

def format_prompt_chatml(code, features, seed, target_intensity=None):
    features_json = json.dumps(features, separators=(',', ':'))
    intensity_line = f'Target intensity: {target_intensity}\n' if target_intensity else ''
    user = (
        f"=== CODE ===\n{code}\n=== END CODE ===\n\n"
        f"=== AST FEATURES ===\n{features_json}\n=== END AST FEATURES ===\n\n"
        f"complexity_class={_complexity_class(features.get('cyclomatic_complexity', 1))} "
        f"(cyclomatic_complexity={features.get('cyclomatic_complexity', 1)})\n"
        f"{intensity_line}"
        f"seed={seed}\n\nGenerate the obfuscation plan JSON:"
    )
    return (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n")

PROMPT_STYLE = 'inst'  # переопределяется probe-ячейкой

def format_prompt(code, features, seed, target_intensity=None):
    fmt = format_prompt_chatml if PROMPT_STYLE == 'chatml' else format_prompt_inst
    return fmt(code, features, seed, target_intensity)

def extract_json(text):
    text = text.strip()
    if text.startswith('{'):
        try: return json.loads(text)
        except json.JSONDecodeError: pass
    decoder = json.JSONDecoder()
    for start, char in enumerate(text):
        if char != '{': continue
        try:
            obj, _ = decoder.raw_decode(text[start:])
            if 'transforms' in obj or 'order' in obj: return obj
        except json.JSONDecodeError: continue
    return None

def validate_plan_schema(plan):
    if not isinstance(plan, dict) or isinstance(plan, list): return False
    if not all(k in plan for k in ['intensity', 'transforms', 'order']): return False
    if plan['intensity'] not in {'light', 'medium', 'heavy'}: return False
    if not isinstance(plan['transforms'], dict) or not isinstance(plan['order'], list): return False
    enabled = [n for n in ORDER if plan['transforms'].get(n, {}).get('enabled')]
    return plan['order'] == enabled

def shape_matches(plan, target_intensity):
    """v7.1: контент плана следует Target intensity."""
    if target_intensity is None: return True
    order = set(plan.get('order', []))
    if target_intensity == 'light':
        return order <= {'rename', 'dead_code'}
    if target_intensity == 'medium':
        return 'opaque_predicates' not in order and bool(order - {'rename', 'dead_code'})
    if target_intensity == 'heavy':
        return 'opaque_predicates' in order
    return True

def parse_instruction(instruction):
    """-> (code, features, seed, target_intensity) из текста instruction."""
    code = instruction.split('=== CODE ===\n', 1)[1].split('\n=== END CODE ===', 1)[0]
    feat_json = instruction.split('=== AST FEATURES ===\n', 1)[1].split('\n=== END AST FEATURES ===', 1)[0]
    features = json.loads(feat_json)
    m = re.search(r'seed=(\d+)', instruction)
    seed = int(m.group(1)) if m else random.randint(0, 0xFFFFFFFF)
    return code, features, seed, _target_of(instruction)

def generate_one(prompt, max_new_tokens=256):
    out = llm(prompt, max_tokens=max_new_tokens, temperature=0.0,
              stop=['<|im_end|>', '</s>'], echo=False)
    return out['choices'][0]['text']

def infer_plan(code, features, seed, target_intensity=None, retries=2):
    prompt = format_prompt(code, features, seed, target_intensity)
    last_raw = ''
    for _ in range(retries + 1):
        raw = generate_one(prompt)
        last_raw = raw
        plan = extract_json(raw)
        if plan is not None and validate_plan_schema(plan):
            plan['seed'] = seed
            return plan, raw
    return None, last_raw

print('helpers ready')


In [ ]:
# --- Probe формата промпта (для свободного кода; eval идёт по instruction as-is) ---
import json as _json
with open(f'{DATA_DIR}/test.jsonl', encoding='utf-8') as f:
    records = [_json.loads(l) for l in f if l.strip()]
print('test records:', len(records))

code0, feat0, seed0, tgt0 = parse_instruction(records[0]['instruction'])
probe = {}
for style, fmt in (('inst', format_prompt_inst), ('chatml', format_prompt_chatml)):
    raw = generate_one(fmt(code0, feat0, seed0, tgt0))
    plan = extract_json(raw)
    probe[style] = bool(plan and validate_plan_schema(plan))
    print(f'{style}: json+schema ok = {probe[style]} | raw[:80] = {raw[:80]!r}')

PROMPT_STYLE = 'chatml' if probe['chatml'] and not probe['inst'] else 'inst'
print('PROMPT_STYLE =', PROMPT_STYLE, '| target of record[0]:', tgt0)


In [ ]:
# --- Quick eval: 10 записей (parse/schema/obedience/shape) ---
from tqdm.notebook import tqdm
from collections import Counter

def eval_records(recs):
    st = {'total': 0, 'ok': 0, 'field_obey': 0, 'shape_ok': 0,
          'light_total': 0, 'light_pure': 0,
          'orders': Counter(), 'non_light_orders': Counter(), 'latencies': [], 'failures': []}
    for rec in recs:
        code_val, feat_val, seed_val, tgt = parse_instruction(rec['instruction'])
        t0 = time.time()
        plan, raw = infer_plan(code_val, feat_val, seed_val, tgt)
        st['latencies'].append(time.time() - t0)
        st['total'] += 1
        if plan is None:
            st['failures'].append(raw[:120]); continue
        st['ok'] += 1
        st['orders'][tuple(plan['order'])] += 1
        if tgt is None or plan['intensity'] == tgt:
            st['field_obey'] += 1
        if shape_matches(plan, tgt):
            st['shape_ok'] += 1
        if tgt != 'light':
            st['non_light_orders'][tuple(plan['order'])] += 1
        if tgt == 'light':
            st['light_total'] += 1
            if set(plan['order']) <= {'rename', 'dead_code'}:
                st['light_pure'] += 1
    return st

q = eval_records(records[:10])
n = q['total']
print(f"JSON+schema: {q['ok']}/{n} ({q['ok']/max(n,1):.0%}) | avg {sum(q['latencies'])/max(len(q['latencies']),1):.1f}s")
print(f"field obedience: {q['field_obey']}/{n} | shape obedience: {q['shape_ok']}/{n} | light purity: {q['light_pure']}/{max(q['light_total'],1)}")
for s in q['failures'][:3]: print('FAIL:', s)


In [ ]:
# --- Full eval: все 750 записей последовательно (llama.cpp не батчит) ---
from collections import Counter

t0 = time.time()
raws = []
latencies = []
for rec in tqdm(records, desc='generate'):
    code_val, feat_val, seed_val, tgt = parse_instruction(rec['instruction'])
    t1 = time.time()
    raws.append(generate_one(format_prompt(code_val, feat_val, seed_val, tgt)))
    latencies.append(time.time() - t1)
wall = time.time() - t0

# Метрики из уже сгенерированных ответов (без повторной генерации)
st = {'total': 0, 'ok': 0, 'field_obey': 0, 'shape_ok': 0, 'light_total': 0, 'light_pure': 0,
      'orders': Counter(), 'non_light_orders': Counter(), 'failures': []}
for rec, raw in zip(records, raws):
    _, _, _, tgt = parse_instruction(rec['instruction'])
    st['total'] += 1
    plan = extract_json(raw)
    if plan is None or not validate_plan_schema(plan):
        st['failures'].append(raw[:120]); continue
    st['ok'] += 1
    st['orders'][tuple(plan['order'])] += 1
    if tgt is None or plan['intensity'] == tgt:
        st['field_obey'] += 1
    if shape_matches(plan, tgt):
        st['shape_ok'] += 1
    if tgt == 'light':
        st['light_total'] += 1
        if set(plan['order']) <= {'rename', 'dead_code'}:
            st['light_pure'] += 1
    else:
        st['non_light_orders'][tuple(plan['order'])] += 1

total = st['total']
n_light_orders = sum(st['non_light_orders'].values())
top_nl, top_nl_c = (st['non_light_orders'].most_common(1)[0] if st['non_light_orders'] else ((), 0))

full_eval = {
    'total': total,
    'json_schema_ok': st['ok'],
    'field_obedience': st['field_obey'] / max(total, 1),
    'shape_obedience': st['shape_ok'] / max(total, 1),
    'light_purity': (st['light_pure'] / st['light_total']) if st['light_total'] else None,
    'unique_orders': len(st['orders']),
    'top_order_share': max(st['orders'].values()) / max(st['ok'], 1),
    'top_non_light_order': ' > '.join(top_nl),
    'top_non_light_order_share': top_nl_c / max(n_light_orders, 1),
    'avg_latency_seconds': round(sum(latencies) / max(len(latencies), 1), 2),
    'wall_seconds': round(wall, 1),
}
print(f"JSON parse + schema: {st['ok']}/{total} = {st['ok']/total:.1%} (target >= 95%)")
print(f"field obedience:     {st['field_obey']}/{total} = {full_eval['field_obedience']:.1%} (target >= 90%)")
print(f"SHAPE obedience:     {st['shape_ok']}/{total} = {full_eval['shape_obedience']:.1%} (target >= 90%)")
print(f"light purity:        {st['light_pure']}/{st['light_total']} = {full_eval['light_purity']:.1%} (target >= 95%)")
print(f"unique orders:       {full_eval['unique_orders']} (target >= 15)")
print(f"top non-light order: {full_eval['top_non_light_order']} {full_eval['top_non_light_order_share']:.1%} (target <= 45%)")
print(f"avg latency: {full_eval['avg_latency_seconds']}s | wall: {wall/60:.1f} min")
if st['failures']:
    print('failures:', len(st['failures']))
    for s in st['failures'][:3]: print('  FAIL:', s)


In [ ]:
# --- Сохранить eval-отчёт (промежуточный) ---
import json as _json
_json.dump(full_eval, open(f'{OUT_DIR}/eval_v7_report.json', 'w'), indent=2)
print('saved:', f'{OUT_DIR}/eval_v7_report.json')


In [ ]:
# --- Node-движок для semantic pass (engine_bundle_v6.zip: движок не менялся) ---
import shutil, subprocess, os
if shutil.which('node') is None:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'nodejs'], check=False,
                   capture_output=True)
print('node:', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())

if not os.path.exists('/content/nb/engine/index.js'):
    subprocess.run(['unzip', '-o', '-q', ENGINE_ZIP, '-d', '/content/nb'], check=True)
print(sorted(os.listdir('/content/nb')))


In [ ]:
import subprocess

NB_DIR = '/content/nb'

def call_engine(requests, batch_size=100, timeout=600):
    results = []
    for s in range(0, len(requests), batch_size):
        chunk = requests[s:s + batch_size]
        payload = '\n'.join(json.dumps(r) for r in chunk)
        proc = subprocess.run(
            ['node', 'engine/index.js', '--json'],
            input=payload, capture_output=True, text=True,
            cwd=NB_DIR, timeout=timeout)
        if proc.returncode != 0:
            print('engine stderr:', proc.stderr[:500])
        results.extend(json.loads(l) for l in proc.stdout.splitlines() if l.strip())
    return results

sanity = call_engine([{'operation': 'extract_features', 'code': 'function f(a){return a+1;}'}])
print('engine sanity:', sanity[0]['ok'], sanity[0]['value']['cyclomatic_complexity'])


In [ ]:
# --- Прогон семантики на schema-valid планах ---
# Планы берём из full-eval: переиспользуем raws -> plans
plans = []
for rec, raw in zip(records, raws):
    code_val, feat_val, seed_val, tgt = parse_instruction(rec['instruction'])
    plan = extract_json(raw)
    if plan is not None and validate_plan_schema(plan):
        plan['seed'] = seed_val
        plans.append(plan)
    else:
        plans.append(None)

valid = [(i, plans[i]) for i in range(len(records)) if plans[i] is not None]
print('schema-valid планов:', len(valid), 'из', len(records))

apply_reqs = []
for i, plan in valid:
    code_val, _, _, _ = parse_instruction(records[i]['instruction'])
    apply_reqs.append({'operation': 'apply', 'code': code_val, 'plan': plan})

apply_resp = call_engine(apply_reqs)
obfuscated = [r['value']['code'] if r.get('ok') else None for r in apply_resp]

val_reqs = []
for (i, _), obf in zip(valid, obfuscated):
    code_val, _, _, _ = parse_instruction(records[i]['instruction'])
    if obf is not None:
        val_reqs.append({'operation': 'validate', 'original_code': code_val, 'obfuscated_code': obf})
    else:
        val_reqs.append({'operation': 'extract_features', 'code': 'function x(){}'})

val_resp = call_engine(val_reqs)

from collections import Counter
sem = {'total_valid': len(valid), 'apply_failed': 0, 'tests_passed': 0,
       'per_intensity': Counter(), 'per_intensity_total': Counter(),
       'per_shape': Counter(), 'per_shape_total': Counter(), 'reasons': Counter()}
for (i, _), obf, vresp in zip(valid, obfuscated, val_resp):
    intensity = records[i]['metadata'].get('intensity', 'unknown')
    sem['per_intensity_total'][intensity] += 1
    if obf is None:
        sem['apply_failed'] += 1
        continue
    if vresp.get('ok') and vresp['value'].get('tests_passed'):
        sem['tests_passed'] += 1
        sem['per_intensity'][intensity] += 1
        if shape_matches(plans[i], intensity):
            sem['per_shape'][intensity] += 1
        sem['per_shape_total'][intensity] += 1
    else:
        reason = vresp['value'].get('reason', 'unknown') if vresp.get('ok') else 'engine_error'
        sem['reasons'][reason] += 1

n_valid = sem['total_valid']
sem['apply_ok_rate'] = (n_valid - sem['apply_failed']) / max(n_valid, 1)
sem['semantic_pass_rate'] = sem['tests_passed'] / max(n_valid, 1)
print(f"apply ok:      {n_valid - sem['apply_failed']}/{n_valid} = {sem['apply_ok_rate']:.1%}")
print(f"semantic pass: {sem['tests_passed']}/{n_valid} = {sem['semantic_pass_rate']:.1%}")
print('reasons:', dict(sem['reasons']))
for k in ('light', 'medium', 'heavy'):
    t = sem['per_intensity_total'].get(k, 0)
    if t:
        p = sem['per_intensity'][k]
        ps, pst = sem['per_shape'].get(k, 0), sem['per_shape_total'].get(k, 0)
        print(f'  {k}: {p}/{t} = {p/t:.1%} | shape-ok among passed: {ps}/{max(pst,1)} = {ps/max(pst,1):.1%}')


In [ ]:
# --- Итоговый отчёт: JSON + markdown на Drive ---
report = {
    'model': f'Qwen2.5-Coder-7B q4_k_m GGUF ({GGUF_PATH.rsplit(chr(47), 1)[-1]})',
    'dataset': 'final_v7/test.jsonl (conditional v7.1)',
    'prompt_style': PROMPT_STYLE,
    'n_test': full_eval['total'],
    'json_parse_rate': full_eval['json_schema_ok'] / full_eval['total'],
    'schema_rate': full_eval['json_schema_ok'] / full_eval['total'],
    'field_obedience_rate': full_eval['field_obedience'],
    'shape_obedience_rate': full_eval['shape_obedience'],
    'light_purity': full_eval['light_purity'],
    'semantic_pass_rate': sem['semantic_pass_rate'],
    'apply_ok_rate': sem['apply_ok_rate'],
    'unique_orders': full_eval['unique_orders'],
    'top_order_share': full_eval['top_order_share'],
    'top_non_light_order': full_eval['top_non_light_order'],
    'top_non_light_order_share': full_eval['top_non_light_order_share'],
    'semantic_by_intensity': {k: (sem['per_intensity'][k], sem['per_intensity_total'][k])
                              for k in ('light', 'medium', 'heavy')},
    'semantic_reasons': dict(sem['reasons']),
    'wall_seconds': full_eval['wall_seconds'],
}
json.dump(report, open(f'{OUT_DIR}/eval_v7_report.json', 'w'), indent=2)

md_lines = [
    '# NeuroObfuscator v7.1 — Evaluation Report', '',
    f"- Model: {report['model']}",
    f"- Test set: {report['n_test']} records (conditional)",
    f"- JSON parse rate: **{report['json_parse_rate']:.1%}** (target >= 95%)",
    f"- Schema valid rate: **{report['schema_rate']:.1%}** (target >= 90%)",
    f"- Field obedience: **{report['field_obedience_rate']:.1%}** | Shape obedience: **{report['shape_obedience_rate']:.1%}** (target >= 90%)",
    f"- Light purity: **{report['light_purity']:.1%}** (target >= 95%)",
    f"- Semantic pass rate: **{report['semantic_pass_rate']:.1%}**",
    f"- Unique orders: {report['unique_orders']} | top non-light: `{report['top_non_light_order']}` {report['top_non_light_order_share']:.1%} (target <= 20%)",
    '', '## Semantic pass by intensity', '',
    '| intensity | passed | total | rate |', '|---|---|---|---|',
]
for k in ('light', 'medium', 'heavy'):
    p, t = report['semantic_by_intensity'][k]
    md_lines.append(f'| {k} | {p} | {t} | {p/max(t,1):.1%} |')
open(f'{OUT_DIR}/eval_v7_report.md', 'w').write('\n'.join(md_lines) + '\n')

print('\n'.join(md_lines))
print('\nsaved:', OUT_DIR)


In [ ]:
# --- Demo: один и тот же код при трёх Target intensity ---
MY_CODE = """
function calculateDiscount(price, tier) {
  let discount = 0;
  if (tier === 'gold') {
    discount = price * 0.2;
  } else if (tier === 'silver') {
    discount = price * 0.1;
  } else {
    discount = price * 0.05;
  }
  const total = price - discount;
  return Math.round(total * 100) / 100;
}
"""

feat_resp = call_engine([{'operation': 'extract_features', 'code': MY_CODE}])
assert feat_resp[0]['ok'], feat_resp[0].get('error')
features = feat_resp[0]['value']

for tgt in ('light', 'medium', 'heavy'):
    seed = random.randint(0, 0xFFFFFFFF)
    plan, raw = infer_plan(MY_CODE, features, seed, tgt)
    print(f'=== Target intensity: {tgt} | seed: {seed} ===')
    if plan:
        shape = shape_matches(plan, tgt)
        print(f'order: {" > ".join(plan["order"])} | shape_matches: {shape}')
        obf = call_engine([{'operation': 'apply', 'code': MY_CODE, 'plan': plan}])[0]
        if obf.get('ok'):
            val = call_engine([{'operation': 'validate',
                                'original_code': MY_CODE,
                                'obfuscated_code': obf['value']['code']}])[0]
            print('validation tests_passed:', val['value'].get('tests_passed'))
        else:
            print('apply failed:', obf.get('error'))
    else:
        print('invalid raw:', raw[:200])
    print()
